<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Upload and Execute Scripts on FABRIC Nodes

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook demonstrates how to upload a local script to a FABRIC node, execute it remotely, and download the results. This is one of the most common patterns in FABRIC experiments -- automating node configuration by pushing scripts from your Jupyter environment to your VMs.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. **Upload** a local file from your Jupyter environment to a remote FABRIC node using `node.upload_file()`
2. **Execute** a script on a remote node with arguments using `node.execute()`
3. **Download** output files from a remote node back to your local environment using `node.download_file()`
4. Combine upload, execute, and download into an end-to-end automation workflow

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook to set up your `fabric_rc` and `ssh_config` files
2. Be familiar with creating slices (see the [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb) notebook)

**Tip:** The script `config_script.sh` is included in the same directory as this notebook. You can edit it by clicking [here](./config_script.sh) or by navigating to it in the Jupyter file browser on the left side of your screen.

</div>

## Background: The Upload-Execute-Download Pattern

When running experiments on FABRIC, you often need to configure nodes with custom software, scripts, or data files. Rather than typing commands interactively, a common best practice is to:

1. **Write a script locally** in your Jupyter environment where you can version-control and edit it easily
2. **Upload** the script to the remote FABRIC node
3. **Execute** the script on the node, optionally passing arguments
4. **Download** any output or log files back for analysis


FABlib provides three key methods for this workflow:
- **`node.upload_file(local_path, remote_path)`** -- copies a file from your Jupyter environment to the node
- **`node.execute(command)`** -- runs a shell command on the node and returns `(stdout, stderr)`
- **`node.download_file(local_path, remote_path)`** -- copies a file from the node back to your Jupyter environment

## What We're Building

In this notebook we will create a single compute node with default resources.

<img src="./figs/slice_topology.png" width="40%">


---

## Step 1: Import FABlib and Verify Configuration

Every FABRIC notebook starts by importing the FABlib library and creating a `FablibManager` instance. The `show_config()` call prints your current configuration so you can verify that tokens, keys, and paths are set correctly.

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Create the Experiment Slice

We create a simple slice with a single node. This is the same pattern you learned in the [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb) notebook. The node will serve as our remote target for uploading and executing scripts.

In [ ]:
# Choose a unique name for the slice
slice_name='MySlice132'

# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# Add a single node with default settings (random site, 2 cores, 8 GB RAM)
slice.add_node(name="Node1")

# Submit the slice request -- this blocks until the node is ready (~2-5 min)
slice.submit();

## Step 3: Inspect the Slice

Once the slice is active, verify that everything looks correct by inspecting the slice attributes and listing the nodes.

In [ ]:
# Retrieve the slice by name (useful if reconnecting to an existing slice)
slice = fablib.get_slice(name=slice_name)

# Display slice-level information (state, expiration, project)
slice.show()

# List all nodes in the slice with their details
slice.list_nodes()

## Step 4: Upload the Configuration Script

Now we upload our local `config_script.sh` to the remote node. The `upload_file()` method uses SCP through the FABRIC bastion host to transfer the file.

<div class="fab-warning">

**Tip:** You can edit the script before uploading by clicking [here](./config_script.sh) or by navigating to it in the Jupyter file browser on the left.

</div>

In [ ]:
# Get a reference to the node object
node = slice.get_node(name="Node1")        

# Upload the local file to the remote node.
# First argument:  local path (relative to this notebook's directory)
# Second argument: remote path (relative to the default home directory on the node)
result = node.upload_file('config_script.sh','config_script.sh')

## Step 5: Execute the Script on the Node

After uploading, we make the script executable with `chmod +x` and then run it. We pass additional arguments to the script (package names to install) and redirect stdout to a log file on the remote node.

<div class="fab-info">

**How it works:** The `node.execute()` method opens an SSH connection through the FABRIC bastion host, runs the specified command, and returns `(stdout, stderr)`. By redirecting output to `config.log` with `>>`, the output is captured on the remote node rather than streamed back to this notebook cell.

</div>

In [ ]:
# Define the packages we want the script to install
script_args="net-tools tcpdump vim"

# Get the node reference
node = slice.get_node(name="Node1")

# Make the script executable, then run it with arguments.
# The '>>' operator appends stdout to config.log on the remote node,
# so the output will not appear in this cell.
stdout, stderr = node.execute(f'chmod +x config_script.sh && ./config_script.sh {script_args} >> config.log')    

## Step 6: Download the Output

Now we download the log file from the remote node back to our Jupyter environment. This lets us inspect the results locally, archive them, or process them with Python.

In [ ]:
try:
    node = slice.get_node(name="Node1")        

    # Download the remote log file to a local file.
    # First argument:  local path (where to save on your Jupyter environment)
    # Second argument: remote path (file to retrieve from the node)
    node.download_file('config.log','config.log')
except Exception as e:
    print(f"Exception: {e}")

## Step 7: View the Results

The log file is now available in your Jupyter environment. You can view it by clicking it in the file browser or by running the `cat` command below.

In [ ]:
# Display the contents of the downloaded log file
# (the '!' prefix runs a bash command from within the notebook)
!cat config.log

<div class="fab-success">

**Success!** If you see the script output above, you have successfully uploaded a script to a FABRIC node, executed it remotely, and downloaded the results.

</div>

## Step 8: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `upload_file()` fails with timeout | SSH/bastion connectivity issue | Check your bastion key path in `show_config()` output; retry after a moment |
| `Permission denied` when executing script | Script not marked executable | Ensure `chmod +x` is run before executing the script |
| Script runs but packages fail to install | Node OS does not have the expected package manager | Check node image (e.g., `yum` for CentOS/Rocky, `apt` for Ubuntu) |
| `download_file()` fails | File does not exist on remote node | Verify the remote file path; check that the script created the expected output |
| `config.log` is empty | Script output was not redirected properly | Ensure you use `>>` or `>` to redirect output to the log file |
| Slice stuck in `Configuring` state | Site may be busy or down | Try a different site by specifying `site='SITENAME'` in `add_node()` |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `slice.add_node(name)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `fablib.get_slice(name)` | Retrieve an existing slice by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `slice.get_node(name)` | Get a specific node object by name | [get_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_node) |
| `node.upload_file(local, remote)` | Upload a file from Jupyter to the node | [upload_file](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.upload_file) |
| `node.execute(command)` | Execute a shell command on the node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `node.download_file(local, remote)` | Download a file from the node to Jupyter | [download_file](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.download_file) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

Now that you know how to upload and execute scripts, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Parallel Configuration** | [parallel_config](../parallel_config/parallel_config.ipynb) | Run configuration commands on multiple nodes simultaneously using threads |
| **Post Boot Tasks** | [post_boot_tasks](../post_boot_tasks/post_boot_tasks.ipynb) | Automate script uploads and execution at slice creation time |
| **Post Boot Task Templates** | [post_boot_task_templates](../post_boot_task_templates/post_boot_task_templates.ipynb) | Use Jinja2 templates to pass dynamic node info to post boot scripts |
| **Docker Containers** | [docker_containers](../docker_containers/docker_containers.ipynb) | Deploy and manage Docker containers on FABRIC nodes |
| **Execute Commands** | [execute_commands](../ssh_to_nodes/execute_commands.ipynb) | More ways to run commands, capture output, and use threads |